# Synthetic data generator

# batch generate, with different LLM or a specific llm. 
- user input via gradio: amount of data, choose llm, what kind of data, download, quntazi or not
- output: json

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

In [17]:
system_prompt = """You are a synthetic data generator. Your task is to create realistic and diverse synthetic data based on the provided schema and instructions. Ensure that the generated data adheres to the specified formats and constraints. Follow the instructions carefully and produce high-quality synthetic data samples."""

In [18]:
user_prompt = """Generate {num_samples} samples of synthetic data based on the following schema and instructions: 
Schema:
{schema}
Instructions:
{instructions}
Ensure that the data is realistic and adheres to the specified formats. Provide the output in JSON format."""


In [19]:
def generate_synthetic_data(schema, instructions, num_samples):
    prompt = user_prompt.format(schema=schema, instructions=instructions, num_samples=num_samples)
    
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=1500
    )
    
    return response.choices[0].message.content

In [20]:
# Create Gradio interface
def gradio_generate(schema, instructions, num_samples):
    """Wrapper function for Gradio interface"""
    try:
        result = generate_synthetic_data(schema, instructions, num_samples)
        return result
    except Exception as e:
        return f"Error: {str(e)}"

# Build the Gradio interface
interface = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            label="Schema", 
            placeholder="Enter your data schema (e.g., {'name': 'string', 'age': 'integer', 'email': 'string'})",
            lines=5
        ),
        gr.Textbox(
            label="Instructions", 
            placeholder="Enter specific instructions for data generation (e.g., 'Generate realistic customer data with US phone numbers')",
            lines=3
        ),
        gr.Slider(
            minimum=1, 
            maximum=100, 
            step=1, 
            value=5,
            label="Number of Samples"
        )
    ],
    outputs=gr.Textbox(
        label="Generated Synthetic Data (JSON)",
        lines=20
    ),
    title="Synthetic Data Generator",
    description="Generate synthetic data based on your schema and instructions using GPT-4",
    examples=[
        [
            "{'name': 'string', 'age': 'integer', 'email': 'string', 'city': 'string'}",
            "Generate realistic customer profiles with valid email addresses",
            3
        ]
    ]
)

# Launch the interface
interface.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
